# Baby Name Visualisations in France (1900–2020)

This notebook implements the **3 interactive visualisations** for the Week 2 project, based on the INSEE baby names dataset by department.

| # | Theme | Visualisation type |
|---|---|---|
| 1 | Temporal evolution | Interactive line chart (click on legend) |
| 2 | Regional effect | Choropleth map with name selector |
| 3 | Gender effects | M/F proportion chart with name selector |

**Environment**: Poetry + kernel `visualisation-prenoms` (Python 3.12)

## Poetry Environment

The project uses Poetry for dependency management. The Jupyter kernel `visualisation-prenoms` was registered via:

```bash
# From the project directory
poetry init --no-interaction --name "visualisation-prenoms" --python "^3.12"
poetry add altair pandas geopandas jupyter ipykernel vega_datasets
poetry run python -m ipykernel install --user --name="visualisation-prenoms" --display-name="Python (visualisation-prenoms)"
```

Select the **Python (visualisation-prenoms)** kernel in VS Code before running the cells.

In [17]:
import altair as alt
import pandas as pd
import geopandas as gpd

# Allow Altair to handle large datasets (>5000 rows)
alt.data_transformers.enable('json')

print(f"altair   : {alt.__version__}")
print(f"pandas   : {pd.__version__}")
print(f"geopandas: {gpd.__version__}")

altair   : 6.1.0
pandas   : 3.0.3
geopandas: 1.1.3


## Data Loading and Cleaning

The `dpt2020.csv` file contains ~3.7 M rows. We remove:
- Aggregated rare names (`_PRENOMS_RARES`)
- Unknown departments (`XX`)
- Unknown years (`XXXX`)

In [18]:
# --- Load baby names ---
names = pd.read_csv("dpt2020.csv", sep=";", dtype=str)

# Cleaning
names = names[names['preusuel'] != '_PRENOMS_RARES']
names = names[names['dpt'] != 'XX']
names = names[names['annais'] != 'XXXX']

names['annais'] = names['annais'].astype(int)
names['nombre'] = pd.to_numeric(names['nombre'], errors='coerce')
names['sexe']   = names['sexe'].astype(int)
names.dropna(subset=['nombre'], inplace=True)
names['nombre'] = names['nombre'].astype(int)

# --- Load geographic boundaries ---
depts = gpd.read_file('departements-version-simplifiee.geojson')

print(f"Names loaded : {len(names):,} rows")
print(f"Departments  : {len(depts)}")
names.sample(3)

Names loaded : 3,668,274 rows
Departments  : 96


,sexe,preusuel,annais,dpt,nombre
927844,1,JULIEN,1988,85,77
2422202,2,FABIENNE,1959,30,22
1456808,1,ROLAND,1968,974,15


---
## Visualisation 1 — Temporal Evolution of Baby Names

**Questions:** How do baby names evolve over time? Are some names consistently popular? Did any names experience a sudden, brief peak in popularity? Are there trends over time?

**Design choice:** Two-layer interactive line chart with two independent interactions:

- **Coloured top 20 lines** — the 20 most popular names, each with its own colour, permanently visible.  
  → **Click a name in the legend** to focus on it: all other lines grey out, making it easy to isolate a trajectory.

- **Red search line** — a dropdown lets you search any name from a pool of 500+ (including names with a sharp yearly peak ≥ 300 births, capturing briefly popular names).  
  → The selected name is overlaid in bold red on top of the top 20, allowing direct comparison.

In [19]:
# National aggregation by (name, year)
names_time = names.groupby(['preusuel', 'annais'], as_index=False)['nombre'].sum()

# --- Background layer: top 20 names by cumulative total ---
top20 = (
    names_time.groupby('preusuel')['nombre']
    .sum()
    .nlargest(20)
    .index.tolist()
)
viz1_background = names_time[names_time['preusuel'].isin(top20)].copy()

# --- Search pool: top 500 cumulative + names with a notable yearly peak ---
# A name with a sharp but brief peak may rank low cumulatively but high at its peak year
name_totals = names_time.groupby('preusuel')['nombre'].sum()
name_peaks  = names_time.groupby('preusuel')['nombre'].max()

search_pool = sorted(
    set(name_totals.nlargest(500).index) |
    set(name_peaks[name_peaks >= 300].index)
)
viz1_search = names_time[names_time['preusuel'].isin(search_pool)].copy()

print(f"Background : {len(top20)} names (top 20 cumulative)")
print(f"Search pool: {len(search_pool)} names (top 500 + peak ≥ 300/year)")

Background : 20 names (top 20 cumulative)
Search pool: 1145 names (top 500 + peak ≥ 300/year)


In [26]:
x_ticks = list(range(1900, 2021, 10))

# Pan (click+drag) and zoom (scroll) on both axes
zoom = alt.selection_interval(bind='scales', encodings=['x', 'y'])

# --- Layer 1: top 20 coloured lines + legend click to grey others ---
legend_sel = alt.selection_point(fields=['preusuel'], bind='legend')

top20_layer = (
    alt.Chart(viz1_background)
    .mark_line(point=alt.OverlayMarkDef(size=40))
    .encode(
        x=alt.X(
            'annais:O',
            title='Year',
            axis=alt.Axis(values=x_ticks, labelAngle=-45),
        ),
        y=alt.Y('nombre:Q', title='Number of births'),
        color=alt.Color(
            'preusuel:N',
            title='Name',
            legend=alt.Legend(title='Click to grey others'),
        ),
        opacity=alt.condition(legend_sel, alt.value(1.0), alt.value(0.06)),
        strokeWidth=alt.condition(legend_sel, alt.value(2.5), alt.value(0.8)),
        tooltip=[
            alt.Tooltip('preusuel:N', title='Name'),
            alt.Tooltip('annais:O',   title='Year'),
            alt.Tooltip('nombre:Q',   title='Births', format=','),
        ],
    )
    .add_params(legend_sel, zoom)
)

# --- Layer 2: dropdown to search any name (shown in bold red) ---
dropdown1 = alt.binding_select(options=search_pool, name='Search any name: ')
name_sel1 = alt.selection_point(
    fields=['preusuel'],
    bind=dropdown1,
    value=search_pool[0],
)

search_layer = (
    alt.Chart(viz1_search)
    .mark_line(strokeWidth=3.5, point=alt.OverlayMarkDef(size=70, color='#E45756'))
    .encode(
        x=alt.X(
            'annais:O',
            title='Year',
            axis=alt.Axis(values=x_ticks, labelAngle=-45),
        ),
        y=alt.Y('nombre:Q', title='Number of births'),
        color=alt.value('#E45756'),
        tooltip=[
            alt.Tooltip('preusuel:N', title='Name'),
            alt.Tooltip('annais:O',   title='Year'),
            alt.Tooltip('nombre:Q',   title='Births', format=','),
        ],
    )
    .add_params(name_sel1)
    .transform_filter(name_sel1)
)

chart_viz1 = (top20_layer + search_layer).properties(
    width=860,
    height=420,
    title=alt.TitleParams(
        'Baby name evolution — top 20 coloured (click legend to grey others) + any name in red',
        fontSize=14,
    ),
)

chart_viz1

alt.LayerChart(...)

---
## Visualisation 2 — Regional Effect (Choropleth Map)

**Questions:** Are some names more popular in certain regions? Are popular names uniformly popular across the whole country?

**Design choice:** Choropleth map with two independent controls:
- **Dropdown** — select any of the top 200 names.
- **Year slider** — move from 1900 to 2020 to see how the regional distribution of a name shifts over time.

→ The colour encodes births **per 1,000 births** in the department for the selected year (normalised to remove population-size bias).  
→ Departments with no recorded births for that name in that year appear as absent (informative: the name was not used there that year).

In [ ]:
# Aggregation by (department, name, year)
grouped_geo_year = names.groupby(['dpt', 'preusuel', 'annais'], as_index=False)['nombre'].sum()

# Total births per department per year for normalisation
dept_totals_year = (
    names.groupby(['dpt', 'annais'], as_index=False)['nombre']
    .sum()
    .rename(columns={'nombre': 'total_dept'})
)
grouped_geo_year = grouped_geo_year.merge(dept_totals_year, on=['dpt', 'annais'])
grouped_geo_year['pour_mille'] = (
    grouped_geo_year['nombre'] / grouped_geo_year['total_dept'] * 1000
).round(3)

# Top 200 names for the dropdown
top200 = (
    grouped_geo_year.groupby('preusuel')['nombre']
    .sum()
    .nlargest(200)
    .index.sort_values()
    .tolist()
)

# Attribute lookup table — NO geometry, just the compound key + values.
# Key pattern: "{dpt}_{preusuel}_{annais}"
# Used at render time via transform_calculate + transform_lookup (see chart cell).
viz2_lookup = grouped_geo_year[grouped_geo_year['preusuel'].isin(top200)].copy()
viz2_lookup['key'] = (
    viz2_lookup['dpt'] + '_' +
    viz2_lookup['preusuel'] + '_' +
    viz2_lookup['annais'].astype(str)
)
viz2_lookup = viz2_lookup[['key', 'nombre', 'pour_mille']]

# The 96-row GeoDataFrame `depts` (loaded in cell 5) is used directly as the
# chart data source — Altair serialises it as a GeoJSON FeatureCollection,
# which mark_geoshape can render correctly.

print(f"Viz 2 lookup: {len(viz2_lookup):,} rows | {len(top200)} names")
print(f"Dept source : {len(depts)} department polygons")

Viz 2 flat   : 1,151,368 rows | 200 names | 121 years
Dept lookup  : 96 rows  | geometry type: dict


In [ ]:
# Use alt.param (not selection_point) for the name so the selected value is
# exposed as a plain Vega signal called 'selected_name', which can be used
# directly in transform_calculate expressions.
name_param = alt.param(
    name='selected_name',
    value=top200[0],
    bind=alt.binding_select(options=top200, name='Name: '),
)

year_param = alt.param(
    name='year_val',
    value=1970,
    bind=alt.binding_range(min=1900, max=2020, step=1, name='Year: '),
)

chart_viz2 = (
    # Base data: 96-row GeoDataFrame → Altair serialises as GeoJSON FeatureCollection
    # → mark_geoshape renders each feature correctly.
    alt.Chart(depts[['code', 'nom', 'geometry']])
    .mark_geoshape(stroke='white', strokeWidth=0.5)
    .encode(
        # Departments with no data (unmatched lookup → pour_mille is null) → grey
        color=alt.condition(
            'isValid(datum.pour_mille)',
            alt.Color(
                'pour_mille:Q',
                scale=alt.Scale(scheme='blues'),
                title='Births per 1,000',
                legend=alt.Legend(gradientLength=220),
            ),
            alt.value('lightgrey'),
        ),
        tooltip=[
            alt.Tooltip('nom:N',        title='Department'),
            alt.Tooltip('nombre:Q',     title='Births',          format=','),
            alt.Tooltip('pour_mille:Q', title='Per 1,000 births', format='.2f'),
        ],
    )
    # Step 1: build a compound key per department using the current param values.
    # Vega expression has direct access to signals 'selected_name' and 'year_val'.
    .transform_calculate(
        lookup_key="datum.code + '_' + selected_name + '_' + toString(year_val)",
    )
    # Step 2: attach 'nombre' and 'pour_mille' by matching the compound key in viz2_lookup.
    # Unmatched departments get null values → rendered grey by the condition above.
    .transform_lookup(
        lookup='lookup_key',
        from_=alt.LookupData(viz2_lookup, 'key', ['nombre', 'pour_mille']),
    )
    .add_params(name_param, year_param)
    .project(type='mercator')
    .properties(
        width=680,
        height=580,
        title=alt.TitleParams(
            'Regional distribution of a name by year (births per 1,000 in the department)',
            fontSize=14,
        ),
    )
)

chart_viz2

alt.Chart(...)

---
## Visualisation 3 — Gender Effects (Proportion Chart)

**Questions:** Are there gender effects in the data? Does the popularity of names given to both sexes evolve consistently?

**Design choice:** Stacked 100% area chart showing the M/F breakdown of a name over time.  
→ Dropdown to choose from **unisex names** (given to both sexes at more than 5%).  
→ The dashed line at 50% serves as a visual reference to spot gender switches.

In [29]:
# National aggregation by (name, year, sex)
gender_time = names.groupby(['preusuel', 'annais', 'sexe'], as_index=False)['nombre'].sum()

# Pivot: one column per sex
pivot = (
    gender_time
    .pivot_table(index=['preusuel', 'annais'], columns='sexe', values='nombre', fill_value=0)
    .reset_index()
)
pivot.columns.name = None
pivot = pivot.rename(columns={1: 'male', 2: 'female'})
pivot['total'] = pivot['male'] + pivot['female']

# Identify unisex names (5–95% for either sex, total > 1,000)
mixed_stats = pivot.groupby('preusuel').agg(
    masc=('male', 'sum'),
    fem=('female', 'sum'),
).reset_index()
mixed_stats['total'] = mixed_stats['masc'] + mixed_stats['fem']
mixed_stats['prop_f'] = mixed_stats['fem'] / mixed_stats['total']

mixed_names = (
    mixed_stats[
        (mixed_stats['prop_f'] > 0.05) &
        (mixed_stats['prop_f'] < 0.95) &
        (mixed_stats['total'] > 1000)
    ]
    .nlargest(200, 'total')   # top 200 unisex names by total births
    ['preusuel']
    .sort_values()
    .tolist()
)

# Long format for stacked area
viz3_base = pivot[pivot['preusuel'].isin(mixed_names)].copy()
viz3_long = viz3_base.melt(
    id_vars=['preusuel', 'annais', 'total'],
    value_vars=['male', 'female'],
    var_name='gender',
    value_name='count',
)
viz3_long['proportion'] = (viz3_long['count'] / viz3_long['total']).round(4)

print(f"Unisex names identified: {len(mixed_names)}")

Unisex names identified: 55


In [30]:
# Dropdown — unisex names only
dropdown3 = alt.binding_select(options=mixed_names, name='Name: ')
name_select3 = alt.selection_point(
    fields=['preusuel'],
    bind=dropdown3,
    value=mixed_names[0],
)

color_scale = alt.Scale(
    domain=['male', 'female'],
    range=['#4C72B0', '#DD8452'],
)

# Stacked area 100%
area = (
    alt.Chart(viz3_long)
    .mark_area()
    .encode(
        x=alt.X(
            'annais:O',
            title='Year',
            axis=alt.Axis(values=list(range(1900, 2021, 10)), labelAngle=-45),
        ),
        y=alt.Y(
            'proportion:Q',
            stack='normalize',
            title='Proportion',
            axis=alt.Axis(format='%'),
        ),
        color=alt.Color('gender:N', scale=color_scale, title='Gender'),
        order=alt.Order('gender:N', sort='descending'),
        tooltip=[
            alt.Tooltip('preusuel:N',   title='Name'),
            alt.Tooltip('annais:O',     title='Year'),
            alt.Tooltip('gender:N',     title='Gender'),
            alt.Tooltip('count:Q',      title='Births', format=','),
            alt.Tooltip('proportion:Q', title='Proportion', format='.1%'),
        ],
    )
    .add_params(name_select3)
    .transform_filter(name_select3)
    .properties(width=860, height=380)
)

# Reference line at 50%
import pandas as pd
rule = (
    alt.Chart(pd.DataFrame({'y': [0.5]}))
    .mark_rule(color='white', strokeDash=[6, 3], strokeWidth=1.5)
    .encode(y='y:Q')
)

chart_viz3 = (area + rule).properties(
    title=alt.TitleParams(
        'Male / female breakdown of a name over time (1900–2020)',
        fontSize=14,
    )
)

chart_viz3

alt.LayerChart(...)